In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_id = 'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct'

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,   # MPS doesn't reliably support bfloat16 on all ops yet
    attn_implementation="sdpa",

).to(device)
max_new_tokens = 512


Using device: mps


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


ImportError: This modeling file requires the following packages that were not found in your environment: flash_attn. Run `pip install flash_attn`

In [3]:
import sys
print(sys.executable)

import torch
print(torch.__version__)
print(torch.backends.mps.is_available())

/Users/rojankarki/Projects/watermark-llm-code-quality/watermark-llm-codebase/.venv-deepseek/bin/python
2.14.0
True


In [4]:
import transformers
print(transformers.__version__)

4.39.3


In [ ]:
print(model.dtype)                          # e.g. torch.float16
print(next(model.parameters()).dtype)       # cross-check, same result
print(model.get_memory_footprint() / 1e9, "GB")

torch.bfloat16
torch.bfloat16
13.477093888 GB


In [ ]:
def prepare_query(data, language = "python"):
    language = "python" 
    hint = data["hints"].get(language, "").strip()
    hint_line = f"Hint: {hint}\n" if hint else ""

    query = f"""Write a {language} code for the following task description: {data['prompt_description']}
{hint_line} """
    return query

# Check Prepare Query and Response using a sample from dataset

In [3]:
data = {
      "task_number": 11,
      "prompt_title": "Growth of a Population",
      "prompt_description": "In a small town the population is p0 = 1000 at the beginning of a year. The population regularly increases by 2 percent per year and moreover 50 new inhabitants per year come to live in the town. How many years does the town need to see its population greater than or equal to p = 1200 inhabitants?\n\nAt the end of the first year there will be: \n1000 + 1000 * 0.02 + 50 => 1070 inhabitants\n\nAt the end of the 2nd year there will be: \n1070 + 1070 * 0.02 + 50 => 1141 inhabitants (** number of inhabitants is an integer **)\n\nAt the end of the 3rd year there will be:\n1141 + 1141 * 0.02 + 50 => 1213\n\nIt will need 3 entire years.\nMore generally given parameters:\n\np0, percent, aug (inhabitants coming or leaving each year), p (population to equal or surpass)\n\nthe function nb_year should return n number of entire years needed to get a population greater or equal to p.\n\naug is an integer, percent a positive or null floating number, p0 and p are positive integers (> 0)\n\nExamples:\nnb_year(1500, 5, 100, 5000) -> 15\nnb_year(1500000, 2.5, 10000, 2000000) -> 10\nNote:\nDon't forget to convert the percent parameter as a percentage in the body of your function: if the parameter percent is 2 you have to convert it to 0.02.\n\nThere are no fractions of people. At the end of each year, the population count is an integer: 252.8 people round down to 252 persons.",
      "hints": {
        "java": "",
        "c": "",
        "cpp": "",
        "python": ""
      },
      "solutions": {
        "java": "",
        "c": "",
        "cpp": "",
        "python": ""
      },
      "source": "https://www.codewars.com/dashboard",
      "tags": ["ALGORITHMS"],
      "comments": ""
    }
query = prepare_query(data)
print(query)

NameError: name 'prepare_query' is not defined

In [ ]:
messages = [
    {"role": "user", "content": query}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# inputs = tokenizer(query, return_tensors="pt", add_special_tokens = True).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.1,
        eos_token_id=tokenizer.eos_token_id
    )


# response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
response = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]

print(response)

# MARKLLM Framework

In [ ]:
from watermark.auto_watermark import AutoWatermark
from utils.transformers_config import TransformersConfig

# Transformers config
transformers_config = TransformersConfig(model=model,
                                         tokenizer=tokenizer,
                                         device=device,
                                         max_new_tokens=max_new_tokens,
                                         min_length=230,
                                         do_sample=True,
                                         no_repeat_ngram_size=4,
                                        #  temperature=0.1,
                                         )



In [16]:
# Load watermark algorithm
myWatermark = AutoWatermark.load('SynthID', 
                                 algorithm_config='config/SynthID.json',
                                 transformers_config=transformers_config)

In [17]:
import json

def read_json(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    return dataset

def write_json(filename, json_data):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False)

In [18]:
dataset = read_json('../dataset.json')
print(json.dumps(dataset[0], indent=2))
print(len(dataset))

{
  "task_number": 0,
  "prompt_title": "Locking Mechanism",
  "prompt_description": "You have been tasked with developing a program that processes large files concurrently. The program should allow multiple threads to access and process the files simultaneously, while ensuring data integrity and avoiding deadlocks.\nQuestion:\nWrite a program that implements a concurrent file processing system with the following requirements:\nThe program should allow multiple threads to access and process files in a shared directory.\nEach file should be processed by only one thread at a time to maintain data integrity.",
  "hints": {
    "java": "Ensure that the program does not hold locks while sleeping or performing long-running operations, as this can lead to deadlocks or other concurrency issues.",
    "c": "",
    "cpp": "",
    "python": ""
  },
  "solutions": {
    "java": "",
    "c": "",
    "cpp": "",
    "python": ""
  },
  "source": "",
  "tags": [
    "concurrency",
    "file ",
    "lo

In [19]:
result = []
for data in dataset:
    query = prepare_query(data)
    messages = [
        {"role": "user", "content": query}
    ]

    query = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    watermarked_text = myWatermark.generate_watermarked_text(query)
    unwatermarked_text = myWatermark.generate_unwatermarked_text(query)
    detect_result_watermarked_text = myWatermark.detect_watermark(watermarked_text)
    detect_result_unwatermarked_text = myWatermark.detect_watermark(unwatermarked_text)
    result.append({
        **data,
        "outputs": {
            "watermarked": {
                "content": watermarked_text,
                **detect_result_watermarked_text,
            },
            "unwatermarked":{
                "content": unwatermarked_text,
                **detect_result_unwatermarked_text,
            }
        }
    })

print(json.dumps(result, indent=2))

[
  {
    "task_number": 0,
    "prompt_title": "Locking Mechanism",
    "prompt_description": "You have been tasked with developing a program that processes large files concurrently. The program should allow multiple threads to access and process the files simultaneously, while ensuring data integrity and avoiding deadlocks.\nQuestion:\nWrite a program that implements a concurrent file processing system with the following requirements:\nThe program should allow multiple threads to access and process files in a shared directory.\nEach file should be processed by only one thread at a time to maintain data integrity.",
    "hints": {
      "java": "Ensure that the program does not hold locks while sleeping or performing long-running operations, as this can lead to deadlocks or other concurrency issues.",
      "c": "",
      "cpp": "",
      "python": ""
    },
    "solutions": {
      "java": "",
      "c": "",
      "cpp": "",
      "python": ""
    },
    "source": "",
    "tags": [
 

In [20]:
write_json('../result.json', result)

In [21]:
for i in range(2):
    result = []
    for data in dataset:
        query = prepare_query(data)
        messages = [
            {"role": "user", "content": query}
        ]

        query = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        watermarked_text = myWatermark.generate_watermarked_text(query)
        unwatermarked_text = myWatermark.generate_unwatermarked_text(query)
        detect_result_watermarked_text = myWatermark.detect_watermark(watermarked_text)
        detect_result_unwatermarked_text = myWatermark.detect_watermark(unwatermarked_text)
        result.append({
            **data,
            "outputs": {
                "watermarked": {
                    "content": watermarked_text,
                    **detect_result_watermarked_text,
                },
                "unwatermarked":{
                    "content": unwatermarked_text,
                    **detect_result_unwatermarked_text,
                }
            }
        })
    filename = f"result-{i}.json"
    write_json(filename, result)


KeyboardInterrupt: 

In [47]:
print(result[0]['outputs']['watermarked']['content'])

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: You have been tasked with developing a program that processes large files concurrently. The program should allow multiple threads to access and process the files simultaneously, while ensuring data integrity and avoiding deadlocks.
Question:
Write a program that implements a concurrent file processing system with the following requirements:
The program should allow multiple threads to access and process files in a shared directory.
Each file should be processed by only one thread at a time to maintain data integrity.
 
assistant
To implement a concurrent file processing system with the specified requirements, we can use Python's `threading` module along with the `queue.Queue` class to manage tasks and ensure that each file is processed by only one thread at a time. We will also use a lock to protect critical村落.

Here's a sample implementation:

```pyt

In [48]:
print(result[0]['outputs']['unwatermarked']['content'])

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: You have been tasked with developing a program that processes large files concurrently. The program should allow multiple threads to access and process the files simultaneously, while ensuring data integrity and avoiding deadlocks.
Question:
Write a program that implements a concurrent file processing system with the following requirements:
The program should allow multiple threads to access and process files in a shared directory.
Each file should be processed by only one thread at a time to maintain data integrity.
 
assistant
To implement a concurrent file processing system in Python, we can use the `threading` module along with a lock to ensure that each file is processed by only one thread at a time. Here's an example implementation:

```python
import os
import threading

# Define a function to process a single file
def process_file(file_path, lo

# MISC

In [29]:
import re

def extract_code_block(output: str, language: str = "python") -> str:
    pattern = rf"```{language}\s*\n(.*?)```"
    match = re.search(pattern, output, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""

In [37]:
watermarked_text_code = extract_code_block(result[0]['outputs']['watermarked']['content'])
print(watermarked_text_code)

import os
import threading
from queue import Queue

def process_file(file_path, output_queue):
    with open겥


In [31]:
unwatermarked_text_code = extract_code_block(unwatermarked_text)
print(unwatermarked_text_code)

def reverse_five_or_more(s):
    return ' '.join(word[::-1] if len(word) >= 5 else word for word in s.split())

# Test cases
print(reverse_five_or_more("Hey fellow warriors"))  # Output: "Hey wollef sroirraw"
print(reverse_five_or_more("This is a test"))       # Output: "This is a test"
print(reverse_five_or_more("This is another test")) # Output: "This is rehtona test"


In [33]:
def strip_prompt(output: str, query: str) -> str:
    if output.startswith(query):
        return output[len(query):].strip()
    # fallback: find query anywhere and take everything after it
    idx = output.find(query)
    if idx != -1:
        return output[idx + len(query):].strip()
    return output.strip()

unwatermarked = strip_prompt(unwatermarked_text, query)
print(unwatermarked)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a python code for the following task description: Write a function that takes in a string of one or more words, and returns the same string, but with all words that have five or more letters reversed (Just like the name of this Kata). Strings passed in will consist of only letters and spaces. Spaces will be included only when more than one word is present.

Examples:

"Hey fellow warriors"  --> "Hey wollef sroirraw" 
"This is a test        --> "This is a test" 
"This is another test" --> "This is rehtona test"
Output only the raw python code — no explanations, no markdown code fences, no preamble or postamble. The code must be complete and directly executable when saved to a file 
assistant
```python
def reverse_five_or_more(s):
    return ' '.join(word[::-1] if len(word) >= 5 else word for word in s.split())

# Test cases
print(reverse_five_or_more("Hey fellow warriors"))  # Output: "Hey wollef sroi

# VISUALIZATION

In [34]:
from visualize.font_settings import FontSettings
from visualize.visualizer import DiscreteVisualizer
from visualize.legend_settings import DiscreteLegendSettings
from visualize.page_layout_settings import PageLayoutSettings
from visualize.color_scheme import ColorSchemeForDiscreteVisualization

In [35]:
watermarked_data = myWatermark.get_data_for_visualization(watermarked_text)
unwatermarked_data = myWatermark.get_data_for_visualization(unwatermarked_text)

# Init visualizer
visualizer = DiscreteVisualizer(color_scheme=ColorSchemeForDiscreteVisualization(),
                                font_settings=FontSettings(), 
                                page_layout_settings=PageLayoutSettings(),
                                legend_settings=DiscreteLegendSettings())
# Visualize
watermarked_img = visualizer.visualize(data=watermarked_data, 
                                       show_text=True, 
                                       visualize_weight=True, 
                                       display_legend=True)

unwatermarked_img = visualizer.visualize(data=unwatermarked_data,
                                         show_text=True, 
                                         visualize_weight=True, 
                                         display_legend=True)

In [36]:
from PIL import Image

def side_by_side(img1: Image.Image, img2: Image.Image) -> Image.Image:
    w = img1.width + img2.width
    h = max(img1.height, img2.height)
    combined = Image.new("RGB", (w, h), (255, 255, 255))
    combined.paste(img1, (0, 0))
    combined.paste(img2, (img1.width, 0))
    return combined

combined_img = side_by_side(watermarked_img, unwatermarked_img)
combined_img.show()  # opens in default image viewer